In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

# ---------- 1. Load and tidy column names ----------
sheets = pd.read_excel(r"C:\\Users\\Aswin\\CP_Dataset.xlsx", sheet_name=None)
for name in sheets:
    sheets[name].columns = sheets[name].columns.str.strip().str.replace(r"\s+", "_", regex=True)

tx = sheets["Transactions"]
cust = sheets["Customer_Features"]
stores = sheets["Store_Master"]
monthly = sheets["Store_Monthly_Performance"]
locs = sheets["Location_Features"]
monthly["rent_share"] = monthly["monthly_rent_inr"] / monthly["monthly_revenue_inr"]


In [9]:
# ---------- 2. Basic data checks ----------
print("== DATA CHECKS ==")
print("Transactions:", len(tx), "| duplicate ids:", tx["transaction_id"].duplicated().sum())
print("Negative gross profit transactions:", (tx["estimated_gross_profit_inr"] < 0).sum())
for name, df in sheets.items():
    if df.isna().sum().sum() > 0:
        print("Missing values in", name, ":", df.isna().sum().sum())



== DATA CHECKS ==
Transactions: 10000 | duplicate ids: 0
Negative gross profit transactions: 5
Missing values in Customer_Features : 41
Missing values in Location_Features : 94
Missing values in Location_Model_View : 1054


In [11]:
# ---------- 3. Descriptive statistics ----------
print("\n== DESCRIPTIVE STATISTICS ==")
print(tx[["quantity", "unit_price_inr", "net_sales_inr", "discount_pct",
          "estimated_gross_margin_pct"]].describe().round(2))
print(monthly[["avg_daily_transactions", "monthly_revenue_inr", "rent_share"]].describe().round(2))




== DESCRIPTIVE STATISTICS ==
       quantity  unit_price_inr  net_sales_inr  discount_pct  estimated_gross_margin_pct
count  10000.00        10000.00       10000.00      10000.00                    10000.00
mean       1.62          119.00         185.99          0.04                       31.57
std        0.97           63.16         161.05          0.06                        9.36
min        1.00           17.49          14.37          0.00                       -4.23
25%        1.00           82.09          88.72          0.00                       25.18
50%        1.00          114.53         133.45          0.00                       31.93
75%        2.00          154.22         232.50          0.05                       38.88
max        5.00          302.31        1508.15          0.20                       50.77
       avg_daily_transactions  monthly_revenue_inr  rent_share
count                  720.00               720.00      720.00
mean                   405.37           180

In [12]:
# ---------- 4. Cross-sheet reconciliation ----------
print("\n== RECONCILIATION ==")
spend = tx.groupby("customer_id")["net_sales_inr"].sum()
c = cust.set_index("customer_id")
match = np.isclose(spend.reindex(c.index).fillna(0), c["total_net_spend_inr"], atol=0.05).mean()
print("Customer spend matches Transactions:", round(match, 3))
print("transactions_last_90d > total_transactions:",
      round((cust["transactions_last_90d"] > cust["total_transactions"]).mean(), 2), "(inconsistent column)")
print("Transactions sheet net sales (INR):", round(tx["net_sales_inr"].sum()))
print("Store-month revenue (INR):", round(monthly["monthly_revenue_inr"].sum()), "(different scale: do not combine)")




== RECONCILIATION ==
Customer spend matches Transactions: 1.0
transactions_last_90d > total_transactions: 0.81 (inconsistent column)
Transactions sheet net sales (INR): 1859912
Store-month revenue (INR): 1298648813 (different scale: do not combine)


In [13]:
# ---------- 5. Discounts ----------
by_discount = tx.groupby("discount_pct")[["quantity", "estimated_gross_margin_pct"]].mean()
print("\n== DISCOUNTS ==\n", by_discount.round(2))
print("Spearman discount vs quantity: r = %.3f, p = %.3f" % stats.spearmanr(tx["discount_pct"], tx["quantity"]))




== DISCOUNTS ==
               quantity  estimated_gross_margin_pct
discount_pct                                      
0.00              1.62                       34.33
0.05              1.61                       30.82
0.10              1.64                       26.72
0.15              1.62                       23.05
0.20              1.59                       17.87
Spearman discount vs quantity: r = -0.010, p = 0.323


In [14]:
# ---------- 6. Time, channel, city, membership (significance tests) ----------
by_hour = tx.groupby("transaction_hour").size()
print("\n== DIFFERENCES IN BASKET VALUE (p < 0.05 means a real difference) ==")
for col in ["day_type", "weather", "sales_channel", "city"]:
    groups = [g["net_sales_inr"] for _, g in tx.groupby(col)]
    p = stats.kruskal(*groups).pvalue
    print(col, "p =", round(p, 3))
members = [g["total_net_spend_inr"] for _, g in cust.groupby("membership_status")]
print("membership vs spend p =", round(stats.mannwhitneyu(*members).pvalue, 3))




== DIFFERENCES IN BASKET VALUE (p < 0.05 means a real difference) ==
day_type p = 0.313
weather p = 0.667
sales_channel p = 0.637
city p = 0.483
membership vs spend p = 0.321


In [15]:
# ---------- 7. Store level drivers ----------
store = monthly.groupby("store_id").agg(revenue=("monthly_revenue_inr", "mean"),
                                        profit_rate=("profitable_flag", "mean")).reset_index()
store = store.merge(stores[["store_id", "location_id", "city", "store_format"]], on="store_id")
store = store.merge(locs[["location_id", "daily_footfall_estimate", "transit_nodes_within_500m",
                          "synthetic_demand_index", "monthly_rent_estimate_inr"]], on="location_id")
drivers = ["daily_footfall_estimate", "transit_nodes_within_500m",
           "synthetic_demand_index", "monthly_rent_estimate_inr"]
print("\n== STORE DRIVERS (Spearman, 60 stores) ==")
print(store[drivers + ["revenue", "profit_rate"]].corr(method="spearman")[["revenue", "profit_rate"]]
      .loc[drivers].round(2))
print("\nMedian rent share of revenue (0 = unprofitable, 1 = profitable):")
print(monthly.groupby("profitable_flag")["rent_share"].median().round(3))
print("\nProfit rate by store format:\n", store.groupby("store_format")["profit_rate"].mean().round(2))
print("\nProfit rate by city:\n", store.groupby("city")["profit_rate"].agg(["count", "mean"]).round(2))




== STORE DRIVERS (Spearman, 60 stores) ==
                           revenue  profit_rate
daily_footfall_estimate       0.93         0.74
transit_nodes_within_500m     0.79         0.66
synthetic_demand_index        0.74         0.57
monthly_rent_estimate_inr     0.02        -0.50

Median rent share of revenue (0 = unprofitable, 1 = profitable):
profitable_flag
0    0.257
1    0.080
Name: rent_share, dtype: float64

Profit rate by store format:
 store_format
High street         0.72
Mixed-use           0.73
Office district     0.56
Residential         0.46
Transit-adjacent    0.35
Name: profit_rate, dtype: float64

Profit rate by city:
            count  mean
city                  
Bengaluru     11  0.64
Chennai       10  0.28
Delhi          5  0.83
Hyderabad      6  0.47
Mumbai        12  0.69
Pune          16  0.62


In [16]:
# ---------- 8. Customer segments (recency, frequency, spend) ----------
buyers = cust[cust["total_transactions"] > 0].copy()
rfm = buyers[["recency_days_from_2024_12_31", "total_transactions", "total_net_spend_inr"]]
X = StandardScaler().fit_transform(np.log1p(rfm))
scores = {k: silhouette_score(X, KMeans(k, n_init=10, random_state=42).fit_predict(X)) for k in range(2, 6)}
best_k = max(scores, key=scores.get)
buyers["segment"] = KMeans(best_k, n_init=10, random_state=42).fit_predict(X)
print("\n== CUSTOMER SEGMENTS ==")
print("Silhouette by k:", {k: round(v, 2) for k, v in scores.items()}, "-> best k =", best_k)
print(buyers.groupby("segment")[rfm.columns].median().round(1).assign(customers=buyers["segment"].value_counts()))




== CUSTOMER SEGMENTS ==
Silhouette by k: {2: 0.38, 3: 0.31, 4: 0.3, 5: 0.28} -> best k = 2
         recency_days_from_2024_12_31  total_transactions  total_net_spend_inr  customers
segment                                                                                  
0                                40.0                 5.0                913.6       1538
1                               131.0                 2.0                323.6        921


In [17]:
# ---------- 9. Representativeness: operating stores vs all candidate locations ----------
locs["has_store"] = locs["location_id"].isin(stores["location_id"])
print("\n== REPRESENTATIVENESS (store locations vs others) ==")
for col in drivers + ["convenience_competitors_within_1km"]:
    p = stats.ks_2samp(locs.loc[locs.has_store, col], locs.loc[~locs.has_store, col]).pvalue
    print(col, "p =", round(p, 2))
print(locs.groupby("has_store")["city"].value_counts(normalize=True).unstack(0).round(2))
print("Customer age: mean", round(cust["age"].mean(), 1), "| share 50+:", round((cust["age"] >= 50).mean(), 2))




== REPRESENTATIVENESS (store locations vs others) ==
daily_footfall_estimate p = 0.75
transit_nodes_within_500m p = 0.84
synthetic_demand_index p = 0.42
monthly_rent_estimate_inr p = 0.31
convenience_competitors_within_1km p = 0.7
has_store  False  True 
city                   
Bengaluru   0.23   0.18
Chennai     0.11   0.17
Delhi       0.10   0.08
Hyderabad   0.09   0.10
Mumbai      0.26   0.20
Pune        0.20   0.27
Customer age: mean 31.3 | share 50+: 0.03


In [18]:
# ---------- 10. Outliers (IQR rule) ----------
print("\n== OUTLIERS (above Q3 + 1.5 x IQR) ==")
for label, s in [("net_sales_inr", tx["net_sales_inr"]), ("quantity", tx["quantity"]),
                 ("monthly_revenue_inr", monthly["monthly_revenue_inr"]), ("rent_share", monthly["rent_share"])]:
    q1, q3 = s.quantile([0.25, 0.75])
    print(label, ":", int((s > q3 + 1.5 * (q3 - q1)).sum()))




== OUTLIERS (above Q3 + 1.5 x IQR) ==
net_sales_inr : 729
quantity : 661
monthly_revenue_inr : 16
rent_share : 25


In [19]:
# ---------- 11. Insight log ----------
log = pd.DataFrame({"insight": [
    "Keys and sales/profit arithmetic are consistent; a few transactions have negative gross profit",
    "Customer columns transactions_last_90d and days_since_last_purchase are inconsistent: exclude",
    "Transactions and store-month sheets are on different scales: never combine for revenue",
    "Discounts reduce margin but do not raise quantity",
    "Footfall and transit nodes track store revenue; rent does not",
    "Rent share of revenue separates profitable from unprofitable store-months (partly mechanical)",
    "Transit-adjacent stores are least often profitable",
    "Store sample skews to Pune; each city has only 5 to 16 stores",
    "Customer segments are weak: customers average about 4 transactions"],
    "carried_to": ["D2", "D4", "D4", "D3 summary", "D4", "D4 (leakage)", "D5", "D5", "D3 summary"]})
log.to_csv("insight_log.csv", index=False)
print("\nInsight log saved to insight_log.csv")




Insight log saved to insight_log.csv


In [20]:
# ---------- 12. Charts ----------
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
by_discount["estimated_gross_margin_pct"].plot(kind="bar", ax=ax[0], title="Margin % by discount")
by_hour.plot(kind="bar", ax=ax[1], title="Transactions by hour")
ax[2].scatter(store["daily_footfall_estimate"], store["revenue"])
ax[2].set_title("Revenue vs footfall")
store.groupby("store_format")["profit_rate"].mean().sort_values().plot(kind="barh", ax=ax[3],
                                                                       title="Profit rate by format")
plt.tight_layout()
plt.savefig("simple_eda_charts.png")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Aswin\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

Error in callback <function _draw_all_if_interactive at 0x000001D6AE247C40> (for post_execute), with arguments args (),kwargs {}:


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Aswin\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Aswin\\anaconda3\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\LastResortHE-Regular.ttf'

<Figure size 1800x400 with 4 Axes>